In [ ]:
!git clone https://github.com/OCN006/AI-POWERED-PARKINSON-DISEASE-DETECTION-SYSTEM.git repo
%cd repo

Cloning into 'repo'...
remote: Enumerating objects: 120, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 120 (delta 13), reused 8 (delta 8), pack-reused 88 (from 2)
Receiving objects: 100% (120/120), 377.65 MiB | 22.91 MiB/s, done.
Resolving deltas: 100% (25/25), done.
Updating files: 100% (48/48), done.
/content/repo


In [ ]:
!git fetch origin handwriting feature/sleep

From https://github.com/OCN006/AI-POWERED-PARKINSON-DISEASE-DETECTION-SYSTEM
 * branch            handwriting -> FETCH_HEAD
 * branch            feature/sleep -> FETCH_HEAD


In [ ]:
!git checkout origin/handwriting -- hanwriting
!git checkout origin/feature/sleep -- sleep

In [ ]:
# 3. Verify all 4 modality folders are now side-by-side!
!ls -la

total 36
drwxr-xr-x 8 root root 4096 Nov 22 07:07  .
drwxr-xr-x 1 root root 4096 Nov 22 07:06  ..
drwxr-xr-x 2 root root 4096 Nov 22 07:06  gait
drwxr-xr-x 8 root root 4096 Nov 22 07:08  .git
-rw-r--r-- 1 root root  202 Nov 22 07:06  .gitignore
drwxr-xr-x 3 root root 4096 Nov 22 07:06  handwriting
drwxr-xr-x 3 root root 4096 Nov 22 07:07  hanwriting
drwxr-xr-x 2 root root 4096 Nov 22 07:06  sleep
drwxr-xr-x 3 root root 4096 Nov 22 07:06 'Voice Model'


In [10]:
import os
import sys
import joblib
import numpy as np
import pandas as pd
import torch
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications.efficientnet import preprocess_input

# Add repo folders to Python path so we can import Gait features & Sleep model class
sys.path.append("/content/repo/gait")
sys.path.append("/content/repo/sleep")

from features import extract_window_features      # From your gait/features.py
from main import DualSensorFusionModel           # From teammate's sleep/main.py

# ==============================================================================
# 1. EXACT PATHS TO ALL 4 TRAINED MODELS IN YOUR REPO
# ==============================================================================
PATHS = {
    "gait_model":  "/content/repo/gait/final_gait_pd_model.pkl",
    "hw_dir":      "/content/repo/hanwriting/outputs",
    "voice_model": "/content/repo/Voice Model/voice_outputs/voice_model.pkl",
    "sleep_model": "/content/repo/sleep/bidslab_sleep_model.pth",
}

# Validation ROC-AUC scores of each model (determines how much weight each gets)
MODALITY_AUCS = {
    "gait": 0.8312,         # Your 5-fold CV ROC-AUC
    "handwriting": 0.9420,  # Teammate's EfficientNetB0 external ROC-AUC
    "voice": 0.854,        # Voice ensemble ROC-AUC (check Voice Model/voice_outputs/holdout_results.csv for exact number)
    "sleep": 0.70347,        # Sleep-disruption biomarker weight
}


# ==============================================================================
# 2. INDIVIDUAL PREDICTOR FUNCTIONS (Each returns a probability between 0 and 1)
# ==============================================================================

def predict_gait_proba(txt_filepath):
    """
    WHAT IT DOES: Loads your gait_pd_model.pkl, slices the walking .txt file into
    5-second windows, extracts 56 stride/FFT features, and averages probabilities.
    """
    model = joblib.load(PATHS["gait_model"])
    raw = np.loadtxt(txt_filepath)
    sensors = raw[500:-500, 1:19]
    sensors = sensors / (np.mean(sensors[:, 16] + sensors[:, 17]) + 1e-8)

    windows = [extract_window_features(sensors[s : s + 500]) for s in range(0, len(sensors) - 500, 250)]
    X_new = np.nan_to_num(np.array(windows, dtype=np.float32))
    return float(np.mean(model.predict_proba(X_new)[:, 1]))


def predict_handwriting_proba(image_path, model_type="baseline", n_folds=5):
    """
    WHAT IT DOES: Loads the 5 EfficientNetB0 checkpoints from hanwriting/outputs/,
    resizes the spiral/meander JPG to 224x224, and averages all 5 fold predictions.
    """
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img,)
    img = tf.expand_dims(preprocess_input(img), axis=0)

    fold_probs = []
    for fold in range(n_folds):
        ckpt = os.path.join(PATHS["hw_dir"], f"{model_type}_fold{fold}_best.keras")
        if os.path.exists(ckpt):
            model = keras.models.load_model(ckpt, safe_mode=False)
            fold_probs.append(float(model.predict(img, verbose=0)))

    if not fold_probs:
        raise FileNotFoundError(f"No checkpoints found in {PATHS['hw_dir']}")
    return float(np.mean(fold_probs))


def predict_voice_proba(voice_csv_path, subject_id=None):
    """
    WHAT IT DOES: Loads Voice Model/voice_outputs/voice_model.pkl, selects the
    754 acoustic features, and returns the SVM+LogReg+XGBoost ensemble probability.
    """
    bundle = joblib.load(PATHS["voice_model"])
    model, feature_cols = bundle["model"], bundle["feature_cols"]

    df = pd.read_csv(voice_csv_path)
    if feature_cols not in df.columns:
        df = pd.read_csv(voice_csv_path, skiprows=1)

    if subject_id is not None and "id" in df.columns:
        df = df[df["id"] == subject_id]

    df = df.fillna(df.median(numeric_only=True))
    return float(np.mean(model.predict_proba(df[feature_cols].values.astype(float))[:, 1]))


def predict_sleep_pd_risk(hr_csv_path, motion_csv_path):
    """
    WHAT IT DOES: Loads sleep/bidslab_sleep_model.pth to predict 30-second sleep stages,
    then calculates a Parkinson's sleep-disruption risk score (0 to 1) from Wake%,
    Deep Sleep (N3) deficit, and sleep-stage fragmentation.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def prep_sensor(csv_path, win=400):
        vals = pd.read_csv(csv_path).iloc[:, 0].values.astype(np.float32)
        vals = (vals - np.mean(vals)) / (np.std(vals) + 1e-8)
        wins = [vals[i * win : (i + 1) * win] for i in range(len(vals) // win)]
        return torch.tensor(np.array(wins)).unsqueeze(1)

    def make_seq(t, seq_len=21):
        return torch.stack([
            torch.stack([torch.zeros_like(t) if j < 0 else t[j] for j in range(i - seq_len + 1, i + 1)])
            for i in range(t.shape)
        ])

    hr_t, mo_t = prep_sensor(hr_csv_path), prep_sensor(motion_csv_path)
    m_len = min(len(hr_t), len(mo_t))
    hr_seq, mo_seq = make_seq(hr_t[:m_len]).to(device), make_seq(mo_t[:m_len]).to(device)

    model = DualSensorFusionModel(num_classes=6).to(device)
    model.load_state_dict(torch.load(PATHS["sleep_model"], map_location=device, weights_only=True))
    model.eval()

    with torch.no_grad():
        preds = torch.argmax(model(hr_seq, mo_seq), dim=1).cpu().numpy()

    valid = preds[preds != 5]  # Ignore unscored epochs
    if len(valid) == 0:
        return 0.50

    wake_ratio = float(np.mean(valid == 0))
    n3_deficit = max(0.0, (0.20 - float(np.mean(valid == 3))) / 0.20)
    frag       = float(np.mean(valid[1:] != valid[:-1]))

    risk = 0.40 * min(1.0, wake_ratio / 0.35) + 0.35 * n3_deficit + 0.25 * min(1.0, frag / 0.30)
    return float(np.clip(risk, 0.05, 0.95))


# ==============================================================================
# 3. MULTIMODAL LATE FUSION ENGINE (AUC-WEIGHTED SOFT VOTING)
# ==============================================================================

def run_multimodal_fusion(
    gait_file=None,
    hw_img=None,
    voice_csv=None,
    voice_subject_id=None,
    sleep_hr=None,
    sleep_motion=None,
    threshold=0.50
):
    """
    WHAT IT DOES: Collects probabilities from whichever modalities are provided,
    weights each by (ROC-AUC - 0.50), and computes the final Parkinson's diagnosis.
    """
    scores = {}
    if gait_file:                 scores["gait"] = predict_gait_proba(gait_file)
    if hw_img:                    scores["handwriting"] = predict_handwriting_proba(hw_img)
    if voice_csv:                 scores["voice"] = predict_voice_proba(voice_csv, voice_subject_id)
    if sleep_hr and sleep_motion: scores["sleep"] = predict_sleep_pd_risk(sleep_hr, sleep_motion)

    if not scores:
        raise ValueError("Please provide at least one modality input!")

    weighted_sum, total_weight = 0.0, 0.0
    for mod, prob in scores.items():
        w = max(0.05, MODALITY_AUCS[mod] - 0.50)
        weighted_sum += prob * w
        total_weight += w

    final_prob = weighted_sum / total_weight
    return {
        "Final_PD_Probability": round(final_prob, 4),
        "Diagnosis": "PARKINSON'S DISEASE DETECTED" if final_prob >= threshold else "HEALTHY CONTROL",
        "Individual_Modality_Scores": {k: round(v, 4) for k, v in scores.items()},
    }


In [15]:
# Test 1: Run all 4 modalities together on a sample patient
result = run_multimodal_fusion(
    gait_file="/content/repo/JuPt11_06.txt",                                      # Uploaded Gait .txt

)

print("=== 4-MODALITY FUSION TEST RESULT ===")
for key, value in result.items():
    print(f"{key}: {value}")


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ExtraTreeClassifier from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ExtraTreesClassifier from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.9.0 when using version 1.6.1. This might lead t

=== 4-MODALITY FUSION TEST RESULT ===
Final_PD_Probability: 0.989
Diagnosis: PARKINSON'S DISEASE DETECTED
Individual_Modality_Scores: {'gait': 0.989}
